# Exp3 evaluation: LLM-agent investigations

| Metric | Question | How it is measured |
|---|---|---|
| Faithfulness | Are the agent's claims true to the data? | share of cited values that match what its tools returned |
| Completeness | Does it find the red flags that are present? | share of the applicant's rule-based red flags (`red_flags.parquet`) it cited |
| Consistency | Same answer when run again? | rating agreement, risk-score spread, overlap of cited evidence across runs |
| Discrimination | Does its blind risk score separate defaulters? | ROC-AUC / PR-AUC vs the hidden TARGET (optionally vs a baseline model) |

**How to use:** (1) build the cohort, (2) map red flags to your tool field names, (3) run the agent and point `RUNS`
at its output, then **Run All**. If `RUNS` is empty, a demo with made-up runs is shown.

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "outputs" / "exp3"] if (p / "eval_exp3.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp3 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Cohort: who to investigate
Test applicants only (the Exp1 model never saw them): the riskiest by the Exp1 model plus a random control group.
The controls are needed to measure discrimination. Set `COHORT_FROM` to the Exp1 test scores (CSV path or DataFrame
with `SK_ID_CURR, score`).

In [2]:
COHORT_FROM = None      # e.g. "path/to/exp1_test_scores.csv"
K_FLAGGED, K_CONTROL = 100, 100

if COHORT_FROM is not None:
    cohort = ev.select_cohort(COHORT_FROM, K_FLAGGED, K_CONTROL)
    cohort.to_csv(HERE / "cohort.csv", index=False)
    print(f"Saved cohort.csv: {len(cohort)} applicants")
    display(cohort.group.value_counts().to_frame("applicants"))
else:
    print("COHORT_FROM not set - skipping.")

COHORT_FROM not set - skipping.


## 2. Red flags and your tool fields
The answer key has 21 rules. For completeness, each rule needs the field name(s) **your agent's tools** return as
evidence for it (e.g. `INST_FREQUENT_LATE` -> `["late_rate"]`). Fill `FLAG_MAP` once; rules you leave out can never be
"found".

In [3]:
rules = pd.read_json(HERE / "red_flag_rules.json").set_index("id")
rules

,source_table,concept,rule,threshold
id,,,,
HIGH_ANNUITY_TO_INCOME,application,repayment burden,annuity / income in top 5% (train),0.354033
LOW_EXT_SCORE,application,external credit score,mean EXT_SOURCE in bottom 10% (train),0.304384
RECENT_ID_CHANGE,application,identity change,ID document changed within 1 year,NaN
RECENT_PHONE_CHANGE,application,identity change,phone changed within 30 days,NaN
SHORT_EMPLOYMENT,application,employment,employed less than 1 year,NaN
MANY_ENQUIRIES,application,credit enquiries,bureau enquiries last year in top 5% (train),6.000000
BUREAU_CURRENT_OVERDUE,bureau,current overdue debt,overdue amount at another lender > 0,NaN
BUREAU_SEVERE_DELINQUENCY,bureau_balance,past delinquency,worst monthly status 31+ days past due,NaN
BUREAU_HIGH_UTILISATION,bureau,debt burden,active debt >= 80% of active credit,NaN


In [4]:
FLAG_MAP = {
    # "INST_FREQUENT_LATE": ["late_rate"],
    # "PREV_HIGH_REFUSAL": ["refusal_rate"],
}   # or a path to a JSON file with the same structure

## 3. Agent runs
A list of run dicts, or a `.jsonl` file with one JSON object per line, for example:
```json
{"SK_ID_CURR": 100054, "run": 1, "risk_score": 0.82, "rating": "high",
 "citations": [{"tool": "get_installment_history", "field": "late_rate", "value": 0.21}],
 "tool_outputs": {"get_installment_history": {"late_rate": 0.214, "max_days_late": 41}}}
```
Run each applicant several times (e.g. 3) to measure consistency.

In [5]:
RUNS = None          # list of dicts or path to agent_runs.jsonl
BASELINE = None      # optional: Exp1 scores (CSV path or DataFrame) to compare the agent's risk score with
THRESHOLD = 0.5      # risk score at or above which the agent counts as flagging the applicant
N_BOOT = 1000
SAVE = True

In [6]:
DEMO = RUNS is None
if DEMO:
    import json
    import numpy as np
    rng = np.random.default_rng(42)
    sp = ev.load_splits()
    test = sp[sp.split == "test"]
    ids = list(test[test.TARGET == 1].SK_ID_CURR.head(3)) + list(test[test.TARGET == 0].SK_ID_CURR.head(3))
    RUNS = []
    for k, sid in enumerate(ids):
        tools = {"get_installment_history": {"late_rate": round(0.3 - 0.05 * k, 3), "max_days_late": 40 - 5 * k},
                 "get_previous_applications": {"refusal_rate": 0.5 if k < 3 else 0.0}}
        for run in range(1, 4):
            cites = [{"tool": "get_installment_history", "field": "late_rate", "value": tools["get_installment_history"]["late_rate"]},
                     {"tool": "get_installment_history", "field": "max_days_late", "value": 40 - 5 * k if run < 3 else 99}]
            RUNS.append({"SK_ID_CURR": int(sid), "run": run, "risk_score": float(np.clip(0.8 - 0.1 * k + rng.normal(0, 0.03), 0, 1)),
                         "rating": "high" if k < 3 else "low", "citations": cites, "tool_outputs": tools})
    FLAG_MAP = FLAG_MAP or {"INST_FREQUENT_LATE": ["late_rate"], "INST_SEVERE_LATE": ["max_days_late"],
                            "PREV_HIGH_REFUSAL": ["refusal_rate"]}
    print("RUNS was empty, so this is a DEMO with made-up agent runs. These numbers are NOT results.")

RUNS was empty, so this is a DEMO with made-up agent runs. These numbers are NOT results.


## 4. Results

In [7]:
per_app, summary = ev.evaluate_agent_runs(RUNS, FLAG_MAP or None, THRESHOLD, BASELINE, N_BOOT)
ev.summary_table(summary)

,value
metric,
applicants,6
runs,18
faithfulness_mean,0.833
citations_not_found_share,0.000
citations_mismatch_share,0.167
completeness_mean,0.050
same_rating_all_runs_share,1.000
evidence_jaccard_mean,1.000
risk_score_std_mean,0.020


### Per applicant
One row per investigated applicant (averaged over runs). `missed_flags` lists red flags that no run cited.

In [8]:
per_app.drop(columns=["is_dev"], errors="ignore")

,SK_ID_CURR,n_runs,risk_score,rating,faithfulness,n_red_flags,completeness,missed_flags,same_rating_all_runs,rating_agreement,risk_score_std,risk_score_range,evidence_jaccard,TARGET,split
0,100047,3,0.800152,high,0.833333,0,NaN,,True,1.0,0.022831,0.053713,1.0,1,test
1,100192,3,0.676874,high,0.833333,2,0.00,"LOW_EXT_SCORE, SHORT_EMPLOYMENT",True,1.0,0.037165,0.086748,1.0,1,test
2,100439,3,0.597948,high,0.833333,4,0.25,"BUREAU_HIGH_UTILISATION, INST_WORSENING, LOW_E...",True,1.0,0.005548,0.013322,1.0,1,test
3,100026,3,0.508041,low,0.833333,1,0.00,BUREAU_HIGH_UTILISATION,True,1.0,0.023815,0.051973,1.0,0,test
4,100027,3,0.416608,low,0.833333,1,0.00,BUREAU_HIGH_UTILISATION,True,1.0,0.013125,0.031836,1.0,0,test
5,100029,3,0.285506,low,0.833333,2,0.00,"BUREAU_HIGH_UTILISATION, RECENT_PHONE_CHANGE",True,1.0,0.018112,0.039829,1.0,0,test


## 5. Save

In [9]:
if SAVE and not DEMO:
    import json
    per_app.to_csv(HERE / "exp3_per_applicant.csv", index=False)
    (HERE / "exp3_summary.json").write_text(json.dumps(summary, indent=2))
    print("Saved result tables in", HERE)
else:
    print("Not saved (demo mode or SAVE = False).")

Not saved (demo mode or SAVE = False).
